# pandas: transformation

Transforming data involves modifying a `Series` or `DataFrame` while maintaining the shape of the original object.

In this lesson I'll demonstrate how to leverage the `Series.transform()`, `DataFrame.transform()` and `DataFrameGroupBy.transform()` methods as we continue to explore the Mega Millions lottery data covering the period 2017-2024.

💡 I've adjusted the `IPython` interactive shell's `ast_node_interactivity` setting to ensure that all cell output is displayed. This overrides the default last expression only ("last_expr") display behavior.

In [27]:
import IPython as ipy
import pandas as pd


# Ensure that all interactive cell output is displayed (default="last_expr")
ipy.get_ipython().ast_node_interactivity = "all"

## Retrieve and prep the data

Let's read in the winning numbers data, convert the "draw_date" column dtype from `object` to `datetime64[ns]`, review a summary of the new `DataFrame` named `data`, and inspect its first three rows.

In [28]:
data = pd.read_csv("./data/mega_millions-aggregate-2017_24.csv")
data["draw_date"] = pd.to_datetime(data["draw_date"], format="%Y-%m-%d")

data.info()
data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 684 entries, 0 to 683
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   draw_date        684 non-null    datetime64[ns]
 1   winning_numbers  684 non-null    object        
 2   mega_ball        684 non-null    int64         
 3   mega_ball_mod2   684 non-null    int64         
 4   multiplier       684 non-null    int64         
 5   pick5_1          684 non-null    int64         
 6   pick5_2          684 non-null    int64         
 7   pick5_3          684 non-null    int64         
 8   pick5_4          684 non-null    int64         
 9   pick5_5          684 non-null    int64         
 10  pick5_sum        684 non-null    int64         
 11  pick5_mod2_sum   684 non-null    int64         
dtypes: datetime64[ns](1), int64(10), object(1)
memory usage: 64.3+ KB


,draw_date,winning_numbers,mega_ball,mega_ball_mod2,multiplier,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5,pick5_sum,pick5_mod2_sum
0,2024-05-17,08 17 40 60 70,3,1,2,8,17,40,60,70,195,1
1,2024-05-14,13 19 43 62 64,6,0,3,13,19,43,62,64,201,3
2,2024-05-10,13 22 26 32 65,18,0,4,13,22,26,32,65,158,2
3,2024-05-07,26 28 36 63 66,15,1,3,26,28,36,63,66,219,1
4,2024-05-03,06 13 15 53 56,11,1,2,6,13,15,53,56,143,3


## Transformation

A transformation operation involves returning a modified `DataFrame` or `Series` possessing the same shape as the "source" `DataFrame` or `Series`. A standard use case involves returning a new `DataFrame` or `Series` of mutated or transformed values in order to combine it with the "source" object.

Both the [`DataFrame.transform()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.transform.html) and [`Series.transform()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.transform.html) methods apply a function designed to mutate data along a specified axis.

`Series.transform(func, axis=0, *args, **kwargs)`

`DataFrame.transform(func, axis=0, *args, **kwargs)`

### Standard scores

In the code below I compute the _standard score_ (a.k.a, the _Z-score_) for each "pick5_sum" value. The standard score represents a value's distance from the mean value of the population under observation. Distance is measured by the number of standard deviations (above or below) that separate the value from the mean.

`Z = (X - μ) / σ`

* `X` = value
* `μ` = population mean (mu)
* `σ` = standard deviation (sigma)

The expression `data.loc[:, "pick5_sum"]` resolves to a `Series`. I call its `transform()` method, passing it an anonymous `lambda` function to generate the standard scores.

In [29]:
pick5_sum_zscores = data.loc[:, "pick5_sum"].transform(
    lambda x: (x - x.mean()) / x.std(ddof=0), axis=0
)  # ddof=0 normalize by N (like NumPy)
pick5_sum_zscores.head(3)

0    0.495525
1    0.632157
2   -0.347041
Name: pick5_sum, dtype: float64

Rather than merge the new `Series` of standard scores with `data`, let's insert a new "pick5_sum_zscores" column into `data` between the "pick5_sum" and "pick5_mod2_sum" columns. Easier on the eyes methinks.

In [30]:
data.insert(data.columns.get_loc("pick5_sum") + 1, "pick5_sum_zscore", pick5_sum_zscores)
data.head(5)

,draw_date,winning_numbers,mega_ball,mega_ball_mod2,multiplier,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5,pick5_sum,pick5_sum_zscore,pick5_mod2_sum
0,2024-05-17,08 17 40 60 70,3,1,2,8,17,40,60,70,195,0.495525,1
1,2024-05-14,13 19 43 62 64,6,0,3,13,19,43,62,64,201,0.632157,3
2,2024-05-10,13 22 26 32 65,18,0,4,13,22,26,32,65,158,-0.347041,2
3,2024-05-07,26 28 36 63 66,15,1,3,26,28,36,63,66,219,1.042054,1
4,2024-05-03,06 13 15 53 56,11,1,2,6,13,15,53,56,143,-0.688621,3


Let's check one of the Z-score transformations for correctness. I'll check the first standard score.

In [31]:
z_score = (data["pick5_sum"][0] - data.loc[:, "pick5_sum"].mean()) / data.loc[:, "pick5_sum"].std(
    ddof=0
)
assert pick5_sum_zscores[0] == z_score

You can also perform transformations across a group of columns. The code below transforms each of the "pick5_*" column values to standard scores.

In [32]:
columns = data.columns[5:10]  # pick5_1 to pick5_5
pick5_zscores = data[columns].transform(lambda x: (x - x.mean()) / x.std(ddof=0), axis=0)
pick5_zscores.head(5)

,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5
0,-0.359521,-0.492509,0.431437,1.082709,1.123843
1,0.214504,-0.321895,0.664474,1.241801,0.537490
2,0.214504,-0.065975,-0.656070,-1.144578,0.635216
3,1.706968,0.445864,0.120721,1.321347,0.732941
4,-0.589130,-0.833735,-1.510539,0.525887,-0.244314


Let's again check one of the Z-score transformations for correctness. I'll check the first "pick5_5" standard score.

In [33]:
z_score = (data["pick5_5"][0] - data.loc[:, "pick5_5"].mean()) / data.loc[:, "pick5_5"].std(ddof=0)
assert pick5_zscores["pick5_5"][0] == z_score

Next, let's create a new `DataFrame` that combines both `data` and `pick5_zscores`. Before I concatenate the frames I'll rename the "pick5_zscore" columns by appending a "_zscore" suffix to each column label.

In [34]:
pick5_zscores.rename(columns=lambda x: f"{x}_zscore", inplace=True)
data_zscores = pd.concat([data, pick5_zscores], axis=1)
data_zscores.head(3)

,draw_date,winning_numbers,mega_ball,mega_ball_mod2,multiplier,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5,pick5_sum,pick5_sum_zscore,pick5_mod2_sum,pick5_1_zscore,pick5_2_zscore,pick5_3_zscore,pick5_4_zscore,pick5_5_zscore
0,2024-05-17,08 17 40 60 70,3,1,2,8,17,40,60,70,195,0.495525,1,-0.359521,-0.492509,0.431437,1.082709,1.123843
1,2024-05-14,13 19 43 62 64,6,0,3,13,19,43,62,64,201,0.632157,3,0.214504,-0.321895,0.664474,1.241801,0.537490
2,2024-05-10,13 22 26 32 65,18,0,4,13,22,26,32,65,158,-0.347041,2,0.214504,-0.065975,-0.656070,-1.144578,0.635216



However, I'd prefer a `DataFrame` with a column order that pairs each "pick5_*" and "pick5_\*_zscore" column together. I can achieve this by sorting the frame's columns. This places the "winning_numbers" column in the last position, but I can pop it out and insert it back into its original position. I'll also pop out the "pick5_mod2_sum" column and insert it into the last position in `data_zscores`.

In [35]:
columns = data_zscores.columns.sort_values()  # Sort index
data_zscores = data_zscores[columns]  # Rearrange columns
data_zscores.insert(1, "winning_numbers", data_zscores.pop("winning_numbers"))  # Move column
data_zscores.insert(len(data_zscores.columns) - 1, "pick5_mod2_sum", data_zscores.pop("pick5_mod2_sum"))  # Move column
data_zscores.head(3)

,draw_date,winning_numbers,mega_ball,mega_ball_mod2,multiplier,pick5_1,pick5_1_zscore,pick5_2,pick5_2_zscore,pick5_3,pick5_3_zscore,pick5_4,pick5_4_zscore,pick5_5,pick5_5_zscore,pick5_sum,pick5_sum_zscore,pick5_mod2_sum
0,2024-05-17,08 17 40 60 70,3,1,2,8,-0.359521,17,-0.492509,40,0.431437,60,1.082709,70,1.123843,195,0.495525,1
1,2024-05-14,13 19 43 62 64,6,0,3,13,0.214504,19,-0.321895,43,0.664474,62,1.241801,64,0.537490,201,0.632157,3
2,2024-05-10,13 22 26 32 65,18,0,4,13,0.214504,22,-0.065975,26,-0.656070,32,-1.144578,65,0.635216,158,-0.347041,2


## Group-wise transformations

You can also perform transformations as part of a groupby operation. A number of built-in transformation methods exist that you can chain to a `DataFrame.groupby()` method call. These include, but are not limited to [`bfill()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.bfill.html#pandas.core.groupby.DataFrameGroupBy.bfill), [`cumcount()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.cumcount.html#pandas.core.groupby.DataFrameGroupBy.cumcount), [`diff()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.diff.html#pandas.core.groupby.DataFrameGroupBy.diff), [`ffill()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.ffill.html#pandas.core.groupby.DataFrameGroupBy.ffill), [`pct_change()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.pct_change.html#pandas.core.groupby.DataFrameGroupBy.pct_change), [`rank()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.rank.html#pandas.core.groupby.DataFrameGroupBy.rank), and [`shift()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.shift.html#pandas.core.groupby.DataFrameGroupBy.shift).

In the example below I group on the "draw_date" year segment and then perform a group-wise transformation by calling the [`DataFrameGroupBy.cumcount()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.cumcount.html) method to return an annualized cumulative count of the winning draws for each group in reverse order starting at `1` rather than `0`. I can call the method either directly or indirectly by passing a string alias to the `transform()` method.

In [36]:
annual_counts = (
    data.groupby(data["draw_date"].dt.year, sort=False).cumcount(ascending=False) + 1
)  # start at 1
annual_counts

0      40
1      39
2      38
3      37
4      36
       ..
679     5
680     4
681     3
682     2
683     1
Length: 684, dtype: int64

In [37]:
annual_counts = (
    data.groupby(data["draw_date"].dt.year, sort=False).transform("cumcount", ascending=False) + 1
)  # start at 1
annual_counts

0      40
1      39
2      38
3      37
4      36
       ..
679     5
680     4
681     3
682     2
683     1
Length: 684, dtype: int64

Note the length of the `Series` returned by the method. It's the same length as both `data` and `data_zscores` and the indices are aligned. This allows me to add these annualized cumulative counts to either `DataFrame`.

In [38]:
data_zscores.insert(1, "draw_num", annual_counts)
data_zscores

,draw_date,draw_num,winning_numbers,mega_ball,mega_ball_mod2,multiplier,pick5_1,pick5_1_zscore,pick5_2,pick5_2_zscore,pick5_3,pick5_3_zscore,pick5_4,pick5_4_zscore,pick5_5,pick5_5_zscore,pick5_sum,pick5_sum_zscore,pick5_mod2_sum
0,2024-05-17,40,08 17 40 60 70,3,1,2,8,-0.359521,17,-0.492509,40,0.431437,60,1.082709,70,1.123843,195,0.495525,1
1,2024-05-14,39,13 19 43 62 64,6,0,3,13,0.214504,19,-0.321895,43,0.664474,62,1.241801,64,0.537490,201,0.632157,3
2,2024-05-10,38,13 22 26 32 65,18,0,4,13,0.214504,22,-0.065975,26,-0.656070,32,-1.144578,65,0.635216,158,-0.347041,2
3,2024-05-07,37,26 28 36 63 66,15,1,3,26,1.706968,28,0.445864,36,0.120721,63,1.321347,66,0.732941,219,1.042054,1
4,2024-05-03,36,06 13 15 53 56,11,1,2,6,-0.589130,13,-0.833735,15,-1.510539,53,0.525887,56,-0.244314,143,-0.688621,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
679,2017-11-14,5,01 14 21 22 28,19,1,3,1,-1.163155,14,-0.748429,21,-1.044465,22,-1.940037,28,-2.980627,86,-1.986627,2
680,2017-11-10,4,06 23 38 42 58,24,0,2,6,-0.589130,23,0.019331,38,0.276079,42,-0.349118,58,-0.048863,167,-0.142092,1
681,2017-11-07,3,01 54 60 68 69,11,1,4,1,-1.163155,54,2.663837,60,1.985018,68,1.719076,69,1.026118,252,1.793531,2
682,2017-11-03,2,10 22 42 61 69,3,1,2,10,-0.129911,22,-0.065975,42,0.586795,61,1.162255,69,1.026118,204,0.700473,2


### Write to file

Let's conclude our discussion of `Series`, `DataFrame` and `DataFrameGroupBy` transformations by writing `data_zscores` to a CSV file. We will make use of the data file in a future lesson. 🙂

In [39]:
filepath = "./data/mega_millions-transform-2017_24.csv"
data_zscores.to_csv(filepath, index=False)